In [2]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image
import gradio as gr

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device("mps")

# 데이터 전처리: 이미지를 그대로 사용할 수 없고 텐서로 변경해야 함
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet50_model = torchvision.models.resnet50(
    weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1
)

resnet50_model.fc = nn.Identity()  # .fc > 자동으로 마지막 레이어에 덮어쓰는 방법
resnet50_model = resnet50_model.to(device)

fc_model = nn.Sequential(
    nn.Linear( 2048, 1024 ),
    nn.ReLU(),
    nn.Linear(1024, 1)
)

fc_static_dict = torch.load("fc_model_0.pth", weights_only=True )
fc_model.load_state_dict(fc_static_dict)
fc_model = fc_model.to(device)

model = nn.Sequential(
    resnet50_model,
    fc_model
)
model = model.to(device)
model.eval()

# tire = Image.open("./images/my_tire01.jpg" ) # 내차 타이어 추가할 것
# tire_tensor = preprocess( tire )
# tire_tensor = tire_tensor.unsqueeze(dim=0)
# tire_tensor = tire_tensor.to(device)

# with torch.no_grad():
#     y_pred = torch.sigmoid( model(tire_tensor) )
#     print( y_pred )
#     pass

def predict_image( image_pixels ):
    tire = Image.fromarray( image_pixels )
    tire_tensor = preprocess( tire )
    tire_tensor = tire_tensor.unsqueeze(dim=0)
    tire_tensor = tire_tensor.to(device)
    with torch.no_grad():
        y_pred = torch.sigmoid( model(tire_tensor) )
        y_pred_value = y_pred.item()
        percentage = round( y_pred_value * 100, 3 )
        return f"This tire {percentage}%, remains milleage."

demo = gr.Interface(
    fn=predict_image, 
    inputs=[
        gr.Image()
    ],
    outputs = gr.Text()
)

demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
